#### Faiss

Facebook AI Similarity Search (Faiss) is a library for efficent similarity search and clustering of dense vectors.
It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

In [5]:
loader = TextLoader("sample.txt", encoding="utf8")

documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30)

docs = text_splitter.split_documents(documents)

In [6]:
docs

[Document(metadata={'source': 'sample.txt'}, page_content='The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. \nOur model achieves 28.4 BLEU on the WMT 2014 Englishto-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.0 after training for 3.5 days on eight GPUs, a small fraction of the training costs of t

In [9]:
embeddings = HuggingFaceEmbeddings( model_name="all-MiniLM-L6-v2")

db = FAISS.from_documents(docs, embeddings)

db

C:\Users\teler\AppData\Local\Temp\ipykernel_15244\1815445120.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings( model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2130.41it/s]


In [11]:
## Querying the vector store

query = "What major architectural components, commonly used in existing state-of-the-art models, does the proposed Transformer completely eliminate?"

docs = db.similarity_search(query)

docs[0].page_content

'Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms are used in conjunction with a recurrent network. \n\nIn this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.'

#### As a Retriever

We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other langchain methods, which largely work with retrievers

In [14]:
retriever = db.as_retriever()

docs = retriever.invoke(query)

docs[0].page_content

'Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms are used in conjunction with a recurrent network. \n\nIn this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.'

#### Similarity Seach with score

- There are some FAISS specific methods. 
- One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score o the query to them.
- The returned distance score is L2 distance. Therefore a lower score is better.

In [15]:
docs_and_score = db.similarity_search_with_score(query)

docs_and_score

[(Document(id='e4089db0-7a8a-4920-ba65-6717287ed94e', metadata={'source': 'sample.txt'}, page_content='Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms are used in conjunction with a recurrent network. \n\nIn this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.'),
  np.float32(1.403602)),
 (Document(id='3d870e44-67b4-41e6-a3b2-8ba5b2200882', metadata={'source': 'sample.txt'}, page_content='The dominant sequence transduct

In [16]:
embedding_vector = embeddings.embed_query(query)

embedding_vector

[-0.09858077019453049,
 0.06383999437093735,
 0.03769055753946304,
 -0.03257018327713013,
 0.03410205617547035,
 -0.01988917589187622,
 -0.12191122025251389,
 0.02970905788242817,
 0.012440241873264313,
 -0.03734956309199333,
 -0.015501387417316437,
 0.08045169711112976,
 0.013560199178755283,
 0.02086154744029045,
 0.01769842952489853,
 -0.01654295064508915,
 0.0007438053144142032,
 0.04689257591962814,
 -0.005830664187669754,
 -0.05597079172730446,
 -0.012022984214127064,
 0.015535297803580761,
 -0.0634089857339859,
 -0.015654973685741425,
 0.04268375039100647,
 0.00887590367347002,
 0.0904165506362915,
 -0.07276426255702972,
 -0.03467744216322899,
 -0.025327935814857483,
 -0.015497288666665554,
 0.013700931333005428,
 -0.03289135918021202,
 0.007864264771342278,
 -0.1065363883972168,
 -0.03298868238925934,
 0.044291794300079346,
 -0.03156145289540291,
 -0.03305598348379135,
 -0.034055501222610474,
 0.04854007437825203,
 -0.06522709876298904,
 0.025583699345588684,
 -0.02296533994376

In [17]:
docs_score = db.similarity_search_by_vector(embedding_vector)

docs_score

[Document(id='e4089db0-7a8a-4920-ba65-6717287ed94e', metadata={'source': 'sample.txt'}, page_content='Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms are used in conjunction with a recurrent network. \n\nIn this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.'),
 Document(id='3d870e44-67b4-41e6-a3b2-8ba5b2200882', metadata={'source': 'sample.txt'}, page_content='The dominant sequence transduction models are based on com

In [ ]:
### Saving and loading the vector store

db.save_local("faiss_index") # creates a folder called faiss_index and saves the index and the documents in it

In [23]:
new_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True) # loads the index and the documents from the faiss_index folder

docs = new_db.similarity_search(query)

docs[0].page_content

'Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 16]. In all but a few cases [22], however, such attention mechanisms are used in conjunction with a recurrent network. \n\nIn this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.'